# Difix video testi: hazır model splat render'ını iyileştiriyor mu?

NVIDIA'nın eğitilmiş **Difix** modeli (`nvidia/difix`), splat render videosunun her karesine uygulanır.
Eğitim yok; sadece hazır modelin bizim splat'imizde ne yaptığına bakıyoruz.

**Çıktılar** (Drive'da `MyDrive/<OUT_DIR>/<video adı>/`):
* `side_by_side.mp4`: solda orijinal render, sağda Difix
* `compare.jpg`: 4 kare; modelin en çok değiştirdiği bölgeye zoom (kırmızı kutu)
* `restored.mp4`: sadece düzeltilmiş video
* `summary.md` / `metrics.json`: değişim miktarı, keskinlik, titreme

**Kullanım:** `VIDEO_PATH`'e Drive'daki videonun yolunu yaz (boş bırakırsan dosya yükleme penceresi açılır),
sonra **Runtime → Run all**. GPU runtime gerekli; A100/L4 hızlı, T4 de çalışır (daha yavaş).

**Bilinmesi gerekenler**
* Gerçek fotoğraf olmadığı için doğruluk ölçülemez; Difix bazen detay **uydurur**. Karar zoom'lara bakarak verilir.
* Kareler tek tek işlendiği için videoda hafif titreme olabilir; `summary.md` bunu ayrıca ölçer.
* `difix_ref` modeli için `REF_IMAGE`'e sahnenin gerçek bir fotoğrafını ver (splat'in eğitildiği videodan bir kare).
* Lisans: Difix kodu ve ağırlıkları NVIDIA lisansıyla yalnızca ticari olmayan araştırma/değerlendirme içindir.

In [ ]:
# @title Ayarlar
VIDEO_PATH = ""            # @param {type:"string"}
MODEL = "difix"            # @param ["difix", "difix_ref"]
REF_IMAGE = ""             # @param {type:"string"}
LONG_EDGE = 1024           # @param {type:"integer"}
FRAME_STRIDE = 1           # @param {type:"integer"}
MAX_SECONDS = 0            # @param {type:"number"}
OUT_DIR = "difix_video_test"  # @param {type:"string"}
# VIDEO_PATH örn. "/content/drive/MyDrive/videolar/son_splat.mp4"; boşsa yükleme penceresi açılır.
# LONG_EDGE: işleme çözünürlüğünün uzun kenarı (Difix 1024x576 civarında eğitildi; büyütmez, sadece küçültür).
# FRAME_STRIDE: 2 = her 2. kare (hızlı deneme). MAX_SECONDS: 0 = videonun tamamı.
assert MODEL != "difix_ref" or REF_IMAGE, "difix_ref için REF_IMAGE (gerçek fotoğraf) gerekli; yoksa MODEL='difix' seç."

In [ ]:
# @title Kurulum (sabitlenmiş sürümler)
!pip install -q "diffusers==0.32.2" "transformers==4.47.1" "peft==0.14.0" "accelerate==1.2.1"
!ffmpeg -version | head -1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# @title Kodu yaz (gömülü kaynaklar)
import json, os, sys
from pathlib import Path

SOURCES = json.loads('{"__init__.py": "\\"\\"\\"Splat render restorer pilot (Difix3D-style). See README.md.\\"\\"\\"\\n", "restorer_model.py": "\\"\\"\\"Single-step diffusion restorer (Difix3D architecture).\\n\\nAdapted from NVIDIA Difix3D (https://github.com/nv-tlabs/Difix3D, src/model.py\\nand src/mv_unet.py) under the NVIDIA License in LICENSE-DIFIX3D.txt, which\\nallows non-commercial (research / evaluation) use only.\\n\\nChanges from the original:\\n* works with current diffusers / peft (no pinned 0.25 UNet copy): the\\n  reference-view attention is an attention *processor*, not a forked UNet;\\n* the text prompt is fixed and encoded once;\\n* x0 is computed directly from the scheduler\'s alphas (epsilon or v);\\n* checkpoints contain the full UNet so a saved model is self-contained.\\n\\nHow it works: the degraded image is VAE-encoded and treated as a latent at\\ntimestep ``t`` (default 199). The SD-Turbo UNet predicts the \\"noise\\", x0 is\\nrecovered in one step and decoded. VAE encoder activations are fed to the\\ndecoder through 1x1 skip convs so fine detail survives the round trip. With a\\nreference view, self-attention runs jointly over both views\' tokens.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport torch\\nimport torch.nn.functional as F\\n\\nBASE_MODEL = \\"stabilityai/sd-turbo\\"\\nPROMPT = \\"remove degradation\\"\\nVAE_LORA_TARGETS = (\\n    \\"conv1\\", \\"conv2\\", \\"conv_in\\", \\"conv_shortcut\\", \\"conv\\", \\"conv_out\\",\\n    \\"skip_conv_1\\", \\"skip_conv_2\\", \\"skip_conv_3\\", \\"skip_conv_4\\",\\n    \\"to_k\\", \\"to_q\\", \\"to_v\\", \\"to_out.0\\",\\n)\\n\\n\\ndef _vae_encoder_fwd(self, sample):\\n    sample = self.conv_in(sample)\\n    skips = []\\n    for down_block in self.down_blocks:\\n        skips.append(sample)\\n        sample = down_block(sample)\\n    sample = self.mid_block(sample)\\n    sample = self.conv_norm_out(sample)\\n    sample = self.conv_act(sample)\\n    sample = self.conv_out(sample)\\n    self.current_down_blocks = skips\\n    return sample\\n\\n\\ndef _vae_decoder_fwd(self, sample, latent_embeds=None):\\n    sample = self.conv_in(sample)\\n    upscale_dtype = next(iter(self.up_blocks.parameters())).dtype\\n    sample = self.mid_block(sample, latent_embeds)\\n    sample = sample.to(upscale_dtype)\\n    skip_convs = [self.skip_conv_1, self.skip_conv_2, self.skip_conv_3, self.skip_conv_4]\\n    skips = self.incoming_skip_acts[::-1]\\n    for idx, up_block in enumerate(self.up_blocks):\\n        if not self.ignore_skip:\\n            sample = sample + skip_convs[idx](skips[idx] * self.gamma)\\n        sample = up_block(sample, latent_embeds)\\n    if latent_embeds is None:\\n        sample = self.conv_norm_out(sample)\\n    else:\\n        sample = self.conv_norm_out(sample, latent_embeds)\\n    sample = self.conv_act(sample)\\n    return self.conv_out(sample)\\n\\n\\nclass MultiViewSelfAttnProcessor:\\n    \\"\\"\\"Self-attention over the tokens of all views of a sample.\\n\\n    The UNet sees a batch of (B * V) images; for attention the V views of each\\n    sample are concatenated along the token axis, which lets the degraded view\\n    borrow detail from the reference view.\\n    \\"\\"\\"\\n\\n    def __init__(self):\\n        self.num_views = 1\\n\\n    def __call__(self, attn, hidden_states, encoder_hidden_states=None, attention_mask=None, temb=None, *args, **kwargs):\\n        input_ndim = hidden_states.ndim\\n        if input_ndim == 4:\\n            b0, c, h, w = hidden_states.shape\\n            hidden_states = hidden_states.view(b0, c, h * w).transpose(1, 2)\\n        bv, n, d = hidden_states.shape\\n        v = self.num_views\\n        x = hidden_states.reshape(bv // v, v * n, d) if v > 1 else hidden_states\\n        b = x.shape[0]\\n        q, k, val = attn.to_q(x), attn.to_k(x), attn.to_v(x)\\n        heads = attn.heads\\n        hd = q.shape[-1] // heads\\n        q, k, val = (t.view(b, -1, heads, hd).transpose(1, 2) for t in (q, k, val))\\n        out = F.scaled_dot_product_attention(q, k, val)\\n        out = out.transpose(1, 2).reshape(b, -1, heads * hd).to(q.dtype)\\n        out = attn.to_out[1](attn.to_out[0](out))\\n        if v > 1:\\n            out = out.reshape(bv, n, d)\\n        if input_ndim == 4:\\n            out = out.transpose(-1, -2).reshape(b0, c, h, w)\\n        if getattr(attn, \\"residual_connection\\", False):\\n            out = out + hidden_states\\n        return out / getattr(attn, \\"rescale_output_factor\\", 1.0)\\n\\n\\nclass Restorer(torch.nn.Module):\\n    def __init__(\\n        self,\\n        timestep: int = 199,\\n        lora_rank_vae: int = 4,\\n        device: str = \\"cuda\\",\\n        pretrained: bool = True,\\n        unet_overrides: dict | None = None,\\n        base_model: str = BASE_MODEL,\\n    ):\\n        \\"\\"\\"``pretrained=False`` builds random weights from the configs only, and\\n        ``unet_overrides`` can shrink that random UNet (tests on small GPUs).\\n        ``base_model`` is the Hugging Face repo the parts are loaded from.\\"\\"\\"\\n        super().__init__()\\n        from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel\\n        from diffusers.models.attention_processor import AttnProcessor2_0\\n        from peft import LoraConfig, inject_adapter_in_model\\n        from transformers import AutoTokenizer, CLIPTextConfig, CLIPTextModel\\n\\n        def load(cls, sub, overrides=None):\\n            if pretrained:\\n                return cls.from_pretrained(base_model, subfolder=sub)\\n            config = dict(cls.load_config(base_model, subfolder=sub))\\n            config.update(overrides or {})\\n            return cls.from_config(config)\\n\\n        tokenizer = AutoTokenizer.from_pretrained(base_model, subfolder=\\"tokenizer\\")\\n        if pretrained:\\n            text_encoder = CLIPTextModel.from_pretrained(base_model, subfolder=\\"text_encoder\\")\\n        else:\\n            text_encoder = CLIPTextModel(CLIPTextConfig.from_pretrained(base_model, subfolder=\\"text_encoder\\"))\\n        text_encoder = text_encoder.to(device)\\n        with torch.no_grad():\\n            ids = tokenizer(PROMPT, max_length=tokenizer.model_max_length, padding=\\"max_length\\",\\n                            truncation=True, return_tensors=\\"pt\\").input_ids.to(device)\\n            self.register_buffer(\\"prompt_embeds\\", text_encoder(ids)[0].float(), persistent=False)\\n        del text_encoder\\n\\n        sched = DDPMScheduler.from_pretrained(base_model, subfolder=\\"scheduler\\")\\n        self.prediction_type = sched.config.prediction_type\\n        self.register_buffer(\\"alphas_cumprod\\", sched.alphas_cumprod.float(), persistent=False)\\n        self.timestep = int(timestep)\\n\\n        vae = load(AutoencoderKL, \\"vae\\")\\n        vae.encoder.forward = _vae_encoder_fwd.__get__(vae.encoder, vae.encoder.__class__)\\n        vae.decoder.forward = _vae_decoder_fwd.__get__(vae.decoder, vae.decoder.__class__)\\n        dec = vae.decoder\\n        dec.skip_conv_1 = torch.nn.Conv2d(512, 512, 1, bias=False)\\n        dec.skip_conv_2 = torch.nn.Conv2d(256, 512, 1, bias=False)\\n        dec.skip_conv_3 = torch.nn.Conv2d(128, 512, 1, bias=False)\\n        dec.skip_conv_4 = torch.nn.Conv2d(128, 256, 1, bias=False)\\n        for conv in (dec.skip_conv_1, dec.skip_conv_2, dec.skip_conv_3, dec.skip_conv_4):\\n            torch.nn.init.constant_(conv.weight, 1e-5)\\n        dec.ignore_skip = False\\n        dec.gamma = 1.0\\n        targets = [name for name, _ in dec.named_modules() if name and any(name.endswith(t) for t in VAE_LORA_TARGETS)]\\n        inject_adapter_in_model(\\n            LoraConfig(r=lora_rank_vae, init_lora_weights=\\"gaussian\\", target_modules=targets), dec, adapter_name=\\"vae_skip\\"\\n        )\\n        vae.requires_grad_(False)\\n        self.vae = vae\\n\\n        unet = load(UNet2DConditionModel, \\"unet\\", unet_overrides)\\n        self.mv_processor = MultiViewSelfAttnProcessor()\\n        unet.set_attn_processor(\\n            {name: (self.mv_processor if name.endswith(\\"attn1.processor\\") else AttnProcessor2_0()) for name in unet.attn_processors}\\n        )\\n        self.unet = unet\\n        self.to(device)\\n\\n    @classmethod\\n    def from_difix(cls, repo: str = \\"nvidia/difix\\", device: str = \\"cuda\\") -> \\"Restorer\\":\\n        \\"\\"\\"NVIDIA\'s released Difix weights (\\"nvidia/difix\\" or \\"nvidia/difix_ref\\").\\n\\n        Their VAE (skip convs + \\"vae_skip\\" LoRA) and UNet have exactly this\\n        class\'s parameter names and shapes, so they load strictly.\\n        \\"\\"\\"\\n        from huggingface_hub import hf_hub_download\\n        from safetensors.torch import load_file\\n\\n        model = cls(device=device, base_model=repo)\\n        for part, module in ((\\"vae\\", model.vae), (\\"unet\\", model.unet)):\\n            path = hf_hub_download(repo, f\\"{part}/diffusion_pytorch_model.safetensors\\")\\n            module.load_state_dict(load_file(path), strict=True)\\n        model.set_eval()\\n        return model\\n\\n    # ----- trainable parameter groups -------------------------------------------------\\n    def vae_trainable_parameters(self):\\n        dec = self.vae.decoder\\n        params = [p for n, p in dec.named_parameters() if \\"lora_\\" in n]\\n        for conv in (dec.skip_conv_1, dec.skip_conv_2, dec.skip_conv_3, dec.skip_conv_4):\\n            params += [p for n, p in conv.named_parameters() if \\"lora_\\" not in n]\\n        return params\\n\\n    def set_train(self):\\n        self.unet.train()\\n        self.vae.train()\\n        self.unet.requires_grad_(True)\\n        self.vae.requires_grad_(False)\\n        for p in self.vae_trainable_parameters():\\n            p.requires_grad_(True)\\n\\n    def set_eval(self):\\n        self.unet.eval()\\n        self.vae.eval()\\n        self.requires_grad_(False)\\n\\n    # ----- forward ---------------------------------------------------------------------\\n    def forward(self, x: torch.Tensor, sample_latent: bool = True) -> torch.Tensor:\\n        \\"\\"\\"x: (B, V, 3, H, W) in [-1, 1]; view 0 is the image to restore,\\n        view 1 (optional) the reference. Returns (B, V, 3, H, W) in [-1, 1].\\"\\"\\"\\n        b, v = x.shape[:2]\\n        flat = x.flatten(0, 1)\\n        dist = self.vae.encode(flat).latent_dist\\n        z = (dist.sample() if sample_latent else dist.mode()) * self.vae.config.scaling_factor\\n        self.mv_processor.num_views = v\\n        t = torch.full((z.shape[0],), self.timestep, device=z.device, dtype=torch.long)\\n        cond = self.prompt_embeds.expand(z.shape[0], -1, -1).to(z.dtype)\\n        pred = self.unet(z, t, encoder_hidden_states=cond).sample\\n        a = self.alphas_cumprod[self.timestep].to(z.dtype)\\n        if self.prediction_type == \\"epsilon\\":\\n            z0 = (z - (1 - a).sqrt() * pred) / a.sqrt()\\n        elif self.prediction_type == \\"v_prediction\\":\\n            z0 = a.sqrt() * z - (1 - a).sqrt() * pred\\n        else:  # \\"sample\\"\\n            z0 = pred\\n        self.vae.decoder.incoming_skip_acts = self.vae.encoder.current_down_blocks\\n        out = self.vae.decode(z0 / self.vae.config.scaling_factor).sample.clamp(-1, 1)\\n        return out.view(b, v, *out.shape[1:])\\n\\n    @torch.no_grad()\\n    def restore(self, deg: torch.Tensor, ref: torch.Tensor | None = None) -> torch.Tensor:\\n        \\"\\"\\"deg/ref: (B, 3, H, W) in [0, 1], H and W multiples of 8. Returns [0, 1].\\"\\"\\"\\n        x = deg * 2 - 1\\n        if ref is not None:\\n            x = torch.stack([x, ref * 2 - 1], dim=1)\\n        else:\\n            x = x[:, None]\\n        # bf16 where the GPU supports it (A100/H100/L4); fp32 otherwise (T4), since\\n        # the SD VAE can overflow in fp16.\\n        # (is_bf16_supported() is also True on T4, where bf16 is only emulated and slow.)\\n        bf16 = x.is_cuda and torch.cuda.get_device_capability(x.device)[0] >= 8\\n        with torch.autocast(\\"cuda\\", dtype=torch.bfloat16, enabled=bf16):\\n            out = self.forward(x, sample_latent=False)[:, 0]\\n        return (out.float() * 0.5 + 0.5).clamp(0, 1)\\n\\n    # ----- persistence -----------------------------------------------------------------\\n    def trainable_state_dict(self) -> dict:\\n        vae_keys = {n for n, _ in self.vae.named_parameters() if \\"lora_\\" in n or \\"skip_conv\\" in n}\\n        return {\\n            \\"unet\\": self.unet.state_dict(),\\n            \\"vae\\": {k: v for k, v in self.vae.state_dict().items() if k in vae_keys},\\n            \\"timestep\\": self.timestep,\\n        }\\n\\n    def load_trainable_state_dict(self, sd: dict) -> None:\\n        self.unet.load_state_dict(sd[\\"unet\\"])\\n        missing = self.vae.load_state_dict(sd[\\"vae\\"], strict=False)\\n        bad = [k for k in missing.unexpected_keys]\\n        if bad:\\n            raise KeyError(f\\"Unexpected VAE keys in checkpoint: {bad[:5]}\\")\\n        self.timestep = int(sd.get(\\"timestep\\", self.timestep))\\n", "video.py": "\\"\\"\\"Run a restorer over every frame of a splat render video.\\n\\nThere is no ground-truth photo for a render video, so the outputs are for\\nlooking at (side-by-side video, zoomed comparison sheet) plus three rough\\nnumbers: how much the model changed each frame, how much sharper it got, and\\nwhether frame-to-frame flicker went up (per-frame restoration is not\\ntemporally consistent).\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport shutil\\nimport subprocess\\nimport time\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn.functional as F\\nfrom PIL import Image, ImageDraw\\n\\n\\ndef probe(video: Path) -> dict:\\n    out = subprocess.run(\\n        [\\"ffprobe\\", \\"-v\\", \\"error\\", \\"-select_streams\\", \\"v:0\\", \\"-show_entries\\",\\n         \\"stream=width,height,r_frame_rate,nb_frames:format=duration\\", \\"-of\\", \\"json\\", str(video)],\\n        check=True, capture_output=True, text=True,\\n    ).stdout\\n    info = json.loads(out)\\n    s = info[\\"streams\\"][0]\\n    num, den = (int(x) for x in s[\\"r_frame_rate\\"].split(\\"/\\"))\\n    return {\\n        \\"width\\": int(s[\\"width\\"]),\\n        \\"height\\": int(s[\\"height\\"]),\\n        \\"fps\\": num / den if den else 30.0,\\n        \\"duration\\": float(info.get(\\"format\\", {}).get(\\"duration\\", 0) or 0),\\n    }\\n\\n\\ndef target_size(width: int, height: int, long_edge: int) -> tuple[int, int]:\\n    \\"\\"\\"Scale so the long edge is ``long_edge`` (never upscale); both sides multiples of 8.\\"\\"\\"\\n    s = min(1.0, long_edge / max(width, height))\\n    return max(8, round(width * s / 8) * 8), max(8, round(height * s / 8) * 8)\\n\\n\\ndef extract_frames(video: Path, out_dir: Path, size: tuple[int, int], stride: int = 1, max_seconds: float = 0) -> int:\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    vf = f\\"scale={size[0]}:{size[1]}:flags=lanczos\\"\\n    if stride > 1:\\n        vf = f\\"select=\'not(mod(n\\\\\\\\,{stride}))\',{vf}\\"\\n    cmd = [\\"ffmpeg\\", \\"-v\\", \\"error\\", \\"-y\\", \\"-i\\", str(video)]\\n    if max_seconds > 0:\\n        cmd += [\\"-t\\", str(max_seconds)]\\n    out = str(out_dir / \\"%05d.png\\")\\n    # -fps_mode needs ffmpeg 5.1+; Colab ships 4.4, which only knows -vsync.\\n    try:\\n        subprocess.run(cmd + [\\"-vf\\", vf, \\"-fps_mode\\", \\"vfr\\", out], check=True, capture_output=True)\\n    except subprocess.CalledProcessError:\\n        subprocess.run(cmd + [\\"-vf\\", vf, \\"-vsync\\", \\"vfr\\", out], check=True)\\n    return len(list(out_dir.glob(\\"*.png\\")))\\n\\n\\ndef _load(path: Path, device) -> torch.Tensor:\\n    arr = np.asarray(Image.open(path).convert(\\"RGB\\"), dtype=np.float32) / 255.0\\n    return torch.from_numpy(arr).permute(2, 0, 1)[None].to(device)\\n\\n\\ndef _save(t: torch.Tensor, path: Path) -> None:\\n    arr = (t[0].permute(1, 2, 0).clamp(0, 1).cpu().numpy() * 255 + 0.5).astype(np.uint8)\\n    Image.fromarray(arr).save(path)\\n\\n\\ndef load_ref(path: Path, size: tuple[int, int], device) -> torch.Tensor:\\n    \\"\\"\\"Reference photo, centre-cropped to the frame aspect and resized to the frame size.\\"\\"\\"\\n    img = Image.open(path).convert(\\"RGB\\")\\n    w, h = size\\n    s = max(w / img.width, h / img.height)\\n    img = img.resize((round(img.width * s), round(img.height * s)), Image.LANCZOS)\\n    left, top = (img.width - w) // 2, (img.height - h) // 2\\n    img = img.crop((left, top, left + w, top + h))\\n    return torch.from_numpy(np.asarray(img, dtype=np.float32) / 255.0).permute(2, 0, 1)[None].to(device)\\n\\n\\ndef restore_frames(model, in_dir: Path, out_dir: Path, ref: torch.Tensor | None = None, device=\\"cuda\\", log=print) -> float:\\n    \\"\\"\\"Restore every frame not already in ``out_dir``. Returns seconds per frame.\\"\\"\\"\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    todo = [p for p in sorted(in_dir.glob(\\"*.png\\")) if not (out_dir / p.name).exists()]\\n    if not todo:\\n        return 0.0\\n    log(f\\"[video] restoring {len(todo)} frames\\")\\n    t0, n = time.time(), 0\\n    for i, p in enumerate(todo):\\n        x = _load(p, device)\\n        h, w = x.shape[-2:]\\n        # The UNet wants latent sides divisible by 8, i.e. image sides divisible by 64.\\n        ph, pw = (-h) % 64, (-w) % 64\\n        xp = F.pad(x, (0, pw, 0, ph), mode=\\"reflect\\") if ph or pw else x\\n        rp = F.pad(ref, (0, pw, 0, ph), mode=\\"reflect\\") if ref is not None and (ph or pw) else ref\\n        y = model.restore(xp, rp)[..., :h, :w]\\n        _save(y, out_dir / p.name)\\n        n += 1\\n        if i == 0 or (i + 1) % 25 == 0 or i + 1 == len(todo):\\n            log(f\\"[video] {i + 1}/{len(todo)}  {(time.time() - t0) / n:.2f} s/frame\\")\\n    return (time.time() - t0) / max(n, 1)\\n\\n\\ndef encode(frames_dir: Path, fps: float, out: Path) -> None:\\n    subprocess.run(\\n        [\\"ffmpeg\\", \\"-v\\", \\"error\\", \\"-y\\", \\"-framerate\\", f\\"{fps:.6f}\\", \\"-i\\", str(frames_dir / \\"%05d.png\\"),\\n         \\"-c:v\\", \\"libx264\\", \\"-crf\\", \\"16\\", \\"-preset\\", \\"medium\\", \\"-pix_fmt\\", \\"yuv420p\\", str(out)],\\n        check=True,\\n    )\\n\\n\\ndef _label(img: Image.Image, text: str) -> Image.Image:\\n    d = ImageDraw.Draw(img)\\n    pad = max(4, img.height // 120)\\n    box = d.textbbox((0, 0), text)\\n    d.rectangle((0, 0, box[2] + 2 * pad, box[3] + 2 * pad), fill=(0, 0, 0))\\n    d.text((pad, pad), text, fill=(255, 255, 255))\\n    return img\\n\\n\\ndef side_by_side(before_dir: Path, after_dir: Path, out_dir: Path) -> None:\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    for p in sorted(after_dir.glob(\\"*.png\\")):\\n        if (out_dir / p.name).exists():\\n            continue\\n        a = _label(Image.open(before_dir / p.name).convert(\\"RGB\\"), \\"ONCE (render)\\")\\n        b = _label(Image.open(p).convert(\\"RGB\\"), \\"SONRA (Difix)\\")\\n        canvas = Image.new(\\"RGB\\", (a.width + b.width, a.height))\\n        canvas.paste(a, (0, 0))\\n        canvas.paste(b, (a.width, 0))\\n        canvas.save(out_dir / p.name)\\n\\n\\ndef _most_changed_box(a: np.ndarray, b: np.ndarray, size: int) -> tuple[int, int]:\\n    diff = torch.from_numpy(np.abs(a - b).mean(-1))[None, None]\\n    pooled = F.avg_pool2d(diff, size, stride=max(1, size // 4))\\n    idx = int(pooled.flatten().argmax())\\n    ny, nx = pooled.shape[-2:]\\n    step = max(1, size // 4)\\n    return (idx % nx) * step, (idx // nx) * step\\n\\n\\ndef comparison_sheet(before_dir: Path, after_dir: Path, out: Path, rows: int = 4) -> None:\\n    \\"\\"\\"Rows of: before | after | zoomed before | zoomed after, zoomed where the model changed most.\\"\\"\\"\\n    names = sorted(p.name for p in after_dir.glob(\\"*.png\\"))\\n    picks = [names[i] for i in np.linspace(0, len(names) - 1, min(rows, len(names))).round().astype(int)]\\n    tiles = []\\n    for name in picks:\\n        a = Image.open(before_dir / name).convert(\\"RGB\\")\\n        b = Image.open(after_dir / name).convert(\\"RGB\\")\\n        crop = max(32, min(a.width, a.height) // 3)\\n        x, y = _most_changed_box(np.asarray(a, np.float32), np.asarray(b, np.float32), crop)\\n        za = a.crop((x, y, x + crop, y + crop)).resize((a.height, a.height), Image.NEAREST)\\n        zb = b.crop((x, y, x + crop, y + crop)).resize((a.height, a.height), Image.NEAREST)\\n        d = ImageDraw.Draw(a)\\n        d.rectangle((x, y, x + crop, y + crop), outline=(255, 0, 0), width=max(2, a.height // 200))\\n        row = [_label(a, \\"once\\"), _label(b, \\"sonra\\"), _label(za, \\"once (zoom)\\"), _label(zb, \\"sonra (zoom)\\")]\\n        canvas = Image.new(\\"RGB\\", (sum(t.width for t in row), a.height))\\n        xo = 0\\n        for t in row:\\n            canvas.paste(t, (xo, 0))\\n            xo += t.width\\n        tiles.append(canvas)\\n    sheet = Image.new(\\"RGB\\", (tiles[0].width, sum(t.height for t in tiles)))\\n    yo = 0\\n    for t in tiles:\\n        sheet.paste(t, (0, yo))\\n        yo += t.height\\n    if sheet.width > 3200:\\n        sheet = sheet.resize((3200, round(sheet.height * 3200 / sheet.width)), Image.LANCZOS)\\n    sheet.save(out, quality=92)\\n\\n\\ndef _gray(path: Path) -> np.ndarray:\\n    return np.asarray(Image.open(path).convert(\\"L\\"), dtype=np.float32) / 255.0\\n\\n\\ndef _laplacian_var(g: np.ndarray) -> float:\\n    lap = -4 * g[1:-1, 1:-1] + g[:-2, 1:-1] + g[2:, 1:-1] + g[1:-1, :-2] + g[1:-1, 2:]\\n    return float(lap.var())\\n\\n\\ndef metrics(before_dir: Path, after_dir: Path) -> dict:\\n    names = sorted(p.name for p in after_dir.glob(\\"*.png\\"))\\n    change_psnr, sharp_ratio, flick_b, flick_a = [], [], [], []\\n    prev_b = prev_a = None\\n    for name in names:\\n        gb, ga = _gray(before_dir / name), _gray(after_dir / name)\\n        mse = float(((gb - ga) ** 2).mean())\\n        change_psnr.append(10 * np.log10(1.0 / max(mse, 1e-10)))\\n        sharp_ratio.append(_laplacian_var(ga) / max(_laplacian_var(gb), 1e-10))\\n        if prev_b is not None:\\n            flick_b.append(float(np.abs(gb - prev_b).mean()))\\n            flick_a.append(float(np.abs(ga - prev_a).mean()))\\n        prev_b, prev_a = gb, ga\\n    fb, fa = float(np.mean(flick_b)) if flick_b else 0.0, float(np.mean(flick_a)) if flick_a else 0.0\\n    return {\\n        \\"frames\\": len(names),\\n        \\"change_psnr_db\\": float(np.mean(change_psnr)),\\n        \\"sharpness_ratio\\": float(np.median(sharp_ratio)),\\n        \\"flicker_before\\": fb,\\n        \\"flicker_after\\": fa,\\n        \\"flicker_ratio\\": fa / fb if fb > 0 else float(\\"nan\\"),\\n    }\\n\\n\\ndef summary_markdown(m: dict, info: dict) -> str:\\n    fr = m[\\"flicker_ratio\\"]\\n    return \\"\\\\n\\".join([\\n        f\\"## Difix video testi: {info[\'video\']}\\",\\n        \\"\\",\\n        f\\"* Model: `{info[\'repo\']}`\\" + (f\\", referans: `{info[\'ref\']}`\\" if info.get(\\"ref\\") else \\"\\"),\\n        f\\"* İşlenen kare: {m[\'frames\']} ({info[\'size\'][0]}x{info[\'size\'][1]}), {info[\'sec_per_frame\']:.2f} s/kare\\",\\n        \\"\\",\\n        \\"| ölçüm | değer | nasıl okunur |\\",\\n        \\"|---|---|---|\\",\\n        f\\"| değişim (önce↔sonra PSNR) | {m[\'change_psnr_db\']:.1f} dB | 40+ dB: neredeyse dokunmamış, 25-35: belirgin düzeltme, <22: çok şey uydurmuş olabilir |\\",\\n        f\\"| keskinlik oranı | {m[\'sharpness_ratio\']:.2f}x | >1 daha keskin; tek başına kalite demek değil (gürültü de keskinlik sayılır) |\\",\\n        f\\"| titreme oranı | {fr:.2f}x | ~1 iyi; 1.2+ ise kareler arası titreme artmış (kare kare işlemenin bilinen yan etkisi) |\\",\\n        \\"\\",\\n        \\"Gerçek fotoğraf olmadığı için bu sayılar doğruluk ölçmez; asıl karar `side_by_side.mp4` ve \\"\\n        \\"`compare.jpg`\'deki zoom\'lara bakarak verilir (kırmızı kutu: modelin en çok değiştirdiği bölge).\\",\\n    ])\\n\\n\\ndef run(\\n    video: Path,\\n    out_dir: Path,\\n    model,\\n    repo: str,\\n    work_dir: Path = Path(\\"/content/difix_video\\"),\\n    ref_image: Path | None = None,\\n    long_edge: int = 1024,\\n    stride: int = 1,\\n    max_seconds: float = 0,\\n    device: str = \\"cuda\\",\\n    log=print,\\n) -> dict:\\n    video, out_dir = Path(video), Path(out_dir)\\n    info = probe(video)\\n    size = target_size(info[\\"width\\"], info[\\"height\\"], long_edge)\\n    work = Path(work_dir) / f\\"{video.stem}_{size[0]}x{size[1]}_s{stride}_t{max_seconds:g}\\"\\n    before, after = work / \\"before\\", work / f\\"after_{repo.split(\'/\')[-1]}\\"\\n    if not any(before.glob(\\"*.png\\")):\\n        n = extract_frames(video, before, size, stride, max_seconds)\\n        log(f\\"[video] {info[\'width\']}x{info[\'height\']} @ {info[\'fps\']:.2f} fps -> {n} frames at {size[0]}x{size[1]}\\")\\n    ref = load_ref(Path(ref_image), size, device) if ref_image else None\\n    spf = restore_frames(model, before, after, ref=ref, device=device, log=log)\\n    timing = after.parent / f\\"{after.name}_timing.json\\"\\n    if spf:\\n        timing.write_text(json.dumps({\\"sec_per_frame\\": spf}), encoding=\\"utf-8\\")\\n    elif timing.exists():\\n        spf = json.loads(timing.read_text(encoding=\\"utf-8\\"))[\\"sec_per_frame\\"]\\n\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    fps = info[\\"fps\\"] / stride\\n    encode(after, fps, out_dir / \\"restored.mp4\\")\\n    sbs = work / \\"side_by_side\\"\\n    side_by_side(before, after, sbs)\\n    encode(sbs, fps, out_dir / \\"side_by_side.mp4\\")\\n    comparison_sheet(before, after, out_dir / \\"compare.jpg\\")\\n    m = metrics(before, after)\\n    meta = {\\"video\\": video.name, \\"repo\\": repo, \\"ref\\": Path(ref_image).name if ref_image else None,\\n            \\"size\\": size, \\"stride\\": stride, \\"sec_per_frame\\": spf, **m}\\n    (out_dir / \\"metrics.json\\").write_text(json.dumps(meta, indent=2), encoding=\\"utf-8\\")\\n    md = summary_markdown(m, meta)\\n    (out_dir / \\"summary.md\\").write_text(md, encoding=\\"utf-8\\")\\n    log(md)\\n    return {\\"out_dir\\": out_dir, \\"work\\": work, \\"metrics\\": meta}\\n\\n\\ndef cleanup(work: Path) -> None:\\n    shutil.rmtree(work, ignore_errors=True)\\n", "LICENSE-DIFIX3D.txt": "NVIDIA License\\n\\n1. Definitions\\n\\n“Licensor” means any person or entity that distributes its Work.\\n“Work” means (a) the original work of authorship made available under this license, which may include software, documentation, or other files, and (b) any additions to or derivative works  thereof  that are made available under this license.\\nThe terms “reproduce,” “reproduction,” “derivative works,” and “distribution” have the meaning as provided under U.S. copyright law; provided, however, that for the purposes of this license, derivative works shall not include works that remain separable from, or merely link (or bind by name) to the interfaces of, the Work.\\nWorks are “made available” under this license by including in or with the Work either (a) a copyright notice referencing the applicability of this license to the Work, or (b) a copy of this license.\\n\\n2. License Grant\\n\\n2.1 Copyright Grant. Subject to the terms and conditions of this license, each Licensor grants to you a perpetual, worldwide, non-exclusive, royalty-free, copyright license to use, reproduce, prepare derivative works of, publicly display, publicly perform, sublicense and distribute its Work and any resulting derivative works in any form.\\n\\n3. Limitations\\n\\n3.1 Redistribution. You may reproduce or distribute the Work only if (a) you do so under this license, (b) you include a complete copy of this license with your distribution, and (c) you retain without modification any copyright, patent, trademark, or attribution notices that are present in the Work.\\n\\n3.2 Derivative Works. You may specify that additional or different terms apply to the use, reproduction, and distribution of your derivative works of the Work (“Your Terms”) only if (a) Your Terms provide that the use limitation in Section 3.3 applies to your derivative works, and (b) you identify the specific derivative works that are subject to Your Terms. Notwithstanding Your Terms, this license (including the redistribution requirements in Section 3.1) will continue to apply to the Work itself.\\n\\n3.3 Use Limitation. The Work and any derivative works thereof only may be used or intended for use non-commercially. Notwithstanding the foregoing, NVIDIA Corporation and its affiliates may use the Work and any derivative works commercially. As used herein, “non-commercially” means for research or evaluation purposes only.\\n\\n3.4 Patent Claims. If you bring or threaten to bring a patent claim against any Licensor (including any claim, cross-claim or counterclaim in a lawsuit) to enforce any patents that you allege are infringed by any Work, then your rights under this license from such Licensor (including the grant in Section 2.1) will terminate immediately.\\n\\n3.5 Trademarks. This license does not grant any rights to use any Licensor’s or its affiliates’ names, logos, or trademarks, except as necessary to reproduce the notices described in this license.\\n\\n3.6 Termination. If you violate any term of this license, then your rights under this license (including the grant in Section 2.1) will terminate immediately.\\n\\n4. Disclaimer of Warranty.\\n\\nTHE WORK IS PROVIDED “AS IS” WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, EITHER EXPRESS OR IMPLIED, INCLUDING WARRANTIES OR CONDITIONS OF \\nMERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE, TITLE OR NON-INFRINGEMENT. YOU BEAR THE RISK OF UNDERTAKING ANY ACTIVITIES UNDER THIS LICENSE. \\n\\n5. Limitation of Liability.\\n\\nEXCEPT AS PROHIBITED BY APPLICABLE LAW, IN NO EVENT AND UNDER NO LEGAL THEORY, WHETHER IN TORT (INCLUDING NEGLIGENCE), CONTRACT, OR OTHERWISE SHALL ANY LICENSOR BE LIABLE TO YOU FOR DAMAGES, INCLUDING ANY DIRECT, INDIRECT, SPECIAL, INCIDENTAL, OR CONSEQUENTIAL DAMAGES ARISING OUT OF OR RELATED TO THIS LICENSE, THE USE OR INABILITY TO USE THE WORK (INCLUDING BUT NOT LIMITED TO LOSS OF GOODWILL, BUSINESS INTERRUPTION, LOST PROFITS OR DATA, COMPUTER FAILURE OR MALFUNCTION, OR ANY OTHER DAMAGES OR LOSSES), EVEN IF THE LICENSOR HAS BEEN ADVISED OF THE POSSIBILITY OF SUCH DAMAGES.\\n\\nSTABILITY AI COMMUNITY LICENSE AGREEMENT\\n\\nLast Updated: July 5, 2024\\n\\nINTRODUCTION\\nThis Agreement applies to any individual person or entity (“You”, “Your” or “Licensee”) that uses or distributes any portion or element of the Stability AI Materials or Derivative Works thereof for any Research & Non-Commercial or Commercial purpose. Capitalized terms not otherwise defined herein are defined in Section V below.\\n\\nThis Agreement is intended to allow research, non-commercial, and limited commercial uses of the Models free of charge. In order to ensure that certain limited commercial uses of the Models continue to be allowed, this Agreement preserves free access to the Models for people or organizations generating annual revenue of less than US $1,000,000 (or local currency equivalent).\\n\\nBy clicking “I Accept” or by using or distributing or using any portion or element of the Stability Materials or Derivative Works, You agree that You have read, understood and are bound by the terms of this Agreement. If You are acting on behalf of a company, organization or other entity, then “You” includes you and that entity, and You agree that You: (i) are an authorized representative of such entity with the authority to bind such entity to this Agreement, and (ii) You agree to the terms of this Agreement on that entity’s behalf.\\n\\nRESEARCH & NON-COMMERCIAL USE LICENSE\\nSubject to the terms of this Agreement, Stability AI grants You a non-exclusive, worldwide, non-transferable, non-sublicensable, revocable and royalty-free limited license under Stability AI’s intellectual property or other rights owned by Stability AI embodied in the Stability AI Materials to use, reproduce, distribute, and create Derivative Works of, and make modifications to, the Stability AI Materials for any Research or Non-Commercial Purpose. “Research Purpose” means academic or scientific advancement, and in each case, is not primarily intended for commercial advantage or monetary compensation to You or others. “Non-Commercial Purpose” means any purpose other than a Research Purpose that is not primarily intended for commercial advantage or monetary compensation to You or others, such as personal use (i.e., hobbyist) or evaluation and testing.\\n\\nCOMMERCIAL USE LICENSE\\nSubject to the terms of this Agreement (including the remainder of this Section III), Stability AI grants You a non-exclusive, worldwide, non-transferable, non-sublicensable, revocable and royalty-free limited license under Stability AI’s intellectual property or other rights owned by Stability AI embodied in the Stability AI Materials to use, reproduce, distribute, and create Derivative Works of, and make modifications to, the Stability AI Materials for any Commercial Purpose. “Commercial Purpose” means any purpose other than a Research Purpose or Non-Commercial Purpose that is primarily intended for commercial advantage or monetary compensation to You or others, including but not limited to, (i) creating, modifying, or distributing Your product or service, including via a hosted service or application programming interface, and (ii) for Your business’s or organization’s internal operations. If You are using or distributing the Stability AI Materials for a Commercial Purpose, You must register with Stability AI at (https://stability.ai/community-license). If at any time You or Your Affiliate(s), either individually or in aggregate, generate more than USD $1,000,000 in annual revenue (or the equivalent thereof in Your local currency), regardless of whether that revenue is generated directly or indirectly from the Stability AI Materials or Derivative Works, any licenses granted to You under this Agreement shall terminate as of such date. You must request a license from Stability AI at (https://stability.ai/enterprise) , which Stability AI may grant to You in its sole discretion. If you receive Stability AI Materials, or any Derivative Works thereof, from a Licensee as part of an integrated end user product, then Section III of this Agreement will not apply to you.\\n\\nGENERAL TERMS\\nYour Research, Non-Commercial, and Commercial License(s) under this Agreement are subject to the following terms. a. Distribution & Attribution. If You distribute or make available the Stability AI Materials or a Derivative Work to a third party, or a product or service that uses any portion of them, You shall: (i) provide a copy of this Agreement to that third party, (ii) retain the following attribution notice within a \\"Notice\\" text file distributed as a part of such copies: \\"This Stability AI Model is licensed under the Stability AI Community License, Copyright © Stability AI Ltd. All Rights Reserved”, and (iii) prominently display “Powered by Stability AI” on a related website, user interface, blogpost, about page, or product documentation. If You create a Derivative Work, You may add your own attribution notice(s) to the “Notice” text file included with that Derivative Work, provided that You clearly indicate which attributions apply to the Stability AI Materials and state in the “Notice” text file that You changed the Stability AI Materials and how it was modified. b. Use Restrictions. Your use of the Stability AI Materials and Derivative Works, including any output or results of the Stability AI Materials or Derivative Works, must comply with applicable laws and regulations (including Trade Control Laws and equivalent regulations) and adhere to the Documentation and Stability AI’s AUP, which is hereby incorporated by reference. Furthermore, You will not use the Stability AI Materials or Derivative Works, or any output or results of the Stability AI Materials or Derivative Works, to create or improve any foundational generative AI model (excluding the Models or Derivative Works). c. Intellectual Property. (i) Trademark License. No trademark licenses are granted under this Agreement, and in connection with the Stability AI Materials or Derivative Works, You may not use any name or mark owned by or associated with Stability AI or any of its Affiliates, except as required under Section IV(a) herein. (ii) Ownership of Derivative Works. As between You and Stability AI, You are the owner of Derivative Works You create, subject to Stability AI’s ownership of the Stability AI Materials and any Derivative Works made by or for Stability AI. (iii) Ownership of Outputs. As between You and Stability AI, You own any outputs generated from the Models or Derivative Works to the extent permitted by applicable law. (iv) Disputes. If You or Your Affiliate(s) institute litigation or other proceedings against Stability AI (including a cross-claim or counterclaim in a lawsuit) alleging that the Stability AI Materials, Derivative Works or associated outputs or results, or any portion of any of the foregoing, constitutes infringement of intellectual property or other rights owned or licensable by You, then any licenses granted to You under this Agreement shall terminate as of the date such litigation or claim is filed or instituted. You will indemnify and hold harmless Stability AI from and against any claim by any third party arising out of or related to Your use or distribution of the Stability AI Materials or Derivative Works in violation of this Agreement. (v) Feedback. From time to time, You may provide Stability AI with verbal and/or written suggestions, comments or other feedback related to Stability AI’s existing or prospective technology, products or services (collectively, “Feedback”). You are not obligated to provide Stability AI with Feedback, but to the extent that You do, You hereby grant Stability AI a perpetual, irrevocable, royalty-free, fully-paid, sub-licensable, transferable, non-exclusive, worldwide right and license to exploit the Feedback in any manner without restriction. Your Feedback is provided “AS IS” and You make no warranties whatsoever about any Feedback. d. Disclaimer Of Warranty. UNLESS REQUIRED BY APPLICABLE LAW, THE STABILITY AI MATERIALS AND ANY OUTPUT AND RESULTS THEREFROM ARE PROVIDED ON AN \\"AS IS\\" BASIS, WITHOUT WARRANTIES OF ANY KIND, EITHER EXPRESS OR IMPLIED, INCLUDING, WITHOUT LIMITATION, ANY WARRANTIES OF TITLE, NON-INFRINGEMENT, MERCHANTABILITY, OR FITNESS FOR A PARTICULAR PURPOSE. YOU ARE SOLELY RESPONSIBLE FOR DETERMINING THE APPROPRIATENESS OR LAWFULNESS OF USING OR REDISTRIBUTING THE STABILITY AI MATERIALS, DERIVATIVE WORKS OR ANY OUTPUT OR RESULTS AND ASSUME ANY RISKS ASSOCIATED WITH YOUR USE OF THE STABILITY AI MATERIALS, DERIVATIVE WORKS AND ANY OUTPUT AND RESULTS. e. Limitation Of Liability. IN NO EVENT WILL STABILITY AI OR ITS AFFILIATES BE LIABLE UNDER ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, TORT, NEGLIGENCE, PRODUCTS LIABILITY, OR OTHERWISE, ARISING OUT OF THIS AGREEMENT, FOR ANY LOST PROFITS OR ANY DIRECT, INDIRECT, SPECIAL, CONSEQUENTIAL, INCIDENTAL, EXEMPLARY OR PUNITIVE DAMAGES, EVEN IF STABILITY AI OR ITS AFFILIATES HAVE BEEN ADVISED OF THE POSSIBILITY OF ANY OF THE FOREGOING. f. Term And Termination. The term of this Agreement will commence upon Your acceptance of this Agreement or access to the Stability AI Materials and will continue in full force and effect until terminated in accordance with the terms and conditions herein. Stability AI may terminate this Agreement if You are in breach of any term or condition of this Agreement. Upon termination of this Agreement, You shall delete and cease use of any Stability AI Materials or Derivative Works. Section IV(d), (e), and (g) shall survive the termination of this Agreement. g. Governing Law. This Agreement will be governed by and constructed in accordance with the laws of the United States and the State of California without regard to choice of law principles, and the UN Convention on Contracts for International Sale of Goods does not apply to this Agreement.\\n\\nDEFINITIONS\\n“Affiliate(s)” means any entity that directly or indirectly controls, is controlled by, or is under common control with the subject entity; for purposes of this definition, “control” means direct or indirect ownership or control of more than 50% of the voting interests of the subject entity.\\n\\n\\"Agreement\\" means this Stability AI Community License Agreement.\\n\\n“AUP” means the Stability AI Acceptable Use Policy available at (https://stability.ai/use-policy), as may be updated from time to time.\\n\\n\\"Derivative Work(s)” means (a) any derivative work of the Stability AI Materials as recognized by U.S. copyright laws and (b) any modifications to a Model, and any other model created which is based on or derived from the Model or the Model’s output, including “fine tune” and “low-rank adaptation” models derived from a Model or a Model’s output, but do not include the output of any Model.\\n\\n“Documentation” means any specifications, manuals, documentation, and other written information provided by Stability AI related to the Software or Models.\\n\\n“Model(s)\\" means, collectively, Stability AI’s proprietary models and algorithms, including machine-learning models, trained model weights and other elements of the foregoing listed on Stability’s Core Models Webpage available at (https://stability.ai/core-models), as may be updated from time to time.\\n\\n\\"Stability AI\\" or \\"we\\" means Stability AI Ltd. and its Affiliates.\\n\\n\\"Software\\" means Stability AI’s proprietary software made available under this Agreement now or in the future.\\n\\n“Stability AI Materials” means, collectively, Stability’s proprietary Models, Software and Documentation (and any portion or combination thereof) made available under this Agreement.\\n\\n“Trade Control Laws” means any applicable U.S. and non-U.S. export control and trade sanctions laws and regulations."}')
pkg = Path("/content/splat_restorer_src/splat_restorer")
pkg.mkdir(parents=True, exist_ok=True)
for name, text in SOURCES.items():
    (pkg / name).write_text(text, encoding="utf-8")
if "/content/splat_restorer_src" not in sys.path:
    sys.path.insert(0, "/content/splat_restorer_src")
print(f"{len(SOURCES)} dosya yazıldı: {pkg}")

In [ ]:
# @title Drive + video
from google.colab import drive
drive.mount("/content/drive")
try:  # nvidia/difix açık bir model; token sadece varsa kullanılır (indirme limiti için).
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

if VIDEO_PATH:
    video_path = Path(VIDEO_PATH)
    assert video_path.exists(), f"Video bulunamadı: {video_path} (Drive'daki tam yolu kontrol et)"
else:
    from google.colab import files
    up = Path("/content/upload"); up.mkdir(exist_ok=True)
    print("Videoyu seç (büyük dosyalarda Drive'a koyup VIDEO_PATH kullanmak daha hızlı):")
    uploaded = files.upload()
    name = next(iter(uploaded))
    video_path = up / name
    video_path.write_bytes(uploaded[name])
ref_path = Path(REF_IMAGE) if REF_IMAGE else None
assert ref_path is None or ref_path.exists(), f"Referans görüntü bulunamadı: {ref_path}"

from splat_restorer import video
info = video.probe(video_path)
print(f"{video_path.name}: {info['width']}x{info['height']}, {info['fps']:.2f} fps, {info['duration']:.1f} s")

In [ ]:
# @title Modeli yükle (ilk seferde ~5 GB indirme, birkaç dakika)
import torch
from splat_restorer.restorer_model import Restorer
assert torch.cuda.is_available(), "GPU yok: Runtime → Change runtime type → GPU"
model = Restorer.from_difix(f"nvidia/{MODEL}", device="cuda")
print(f"nvidia/{MODEL} yüklendi")

In [ ]:
# @title Videoyu işle
out_dir = Path("/content/drive/MyDrive") / OUT_DIR / f"{video_path.stem}_{MODEL}"
result = video.run(video_path, out_dir, model, repo=f"nvidia/{MODEL}", ref_image=ref_path,
                   long_edge=LONG_EDGE, stride=FRAME_STRIDE, max_seconds=MAX_SECONDS)
print("Çıktılar:", out_dir)

In [ ]:
# @title Sonuçlar
import base64
from IPython.display import HTML, Image as IPImage, Markdown, display
display(Markdown((out_dir / "summary.md").read_text(encoding="utf-8")))
display(IPImage(filename=str(out_dir / "compare.jpg"), width=1400))
sbs = out_dir / "side_by_side.mp4"
if sbs.stat().st_size < 40 * 2**20:
    b64 = base64.b64encode(sbs.read_bytes()).decode()
    display(HTML(f'<video controls loop width="1400" src="data:video/mp4;base64,{b64}"></video>'))
else:
    print(f"Video notebook'ta gösterilemeyecek kadar büyük; Drive'dan aç: {sbs}")

## Nasıl karar veririz?

* **Zoom'larda** floater'lar, bulanık bölgeler, splat "lekeleri" temizlenmiş ve detay gerçekçi görünüyorsa: umut verici.
* Detay **uydurulmuş** görünüyorsa (olmayan doku, şekil değişimi) ya da video belirgin titriyorsa: hazır model yetmiyor.
* `summary.md`, `compare.jpg` ve `side_by_side.mp4`'ten birkaç saniyelik bir kesit bana getirmen yeterli.